In [1]:
"""
consolidate_data.py - REFACTORED for SIMPLICITY
Creates two master .npz files with intuitive structure:

consolidated_ieeg.npz:
    data[subj]['c']['L_STN']  # Left STN iEEG during eyes closed
    data[subj]['c']['R_STN']  # Right STN iEEG during eyes closed
    data[subj]['l']['L_STN']  # Left STN iEEG during left hand
    data[subj]['r']['R_STN']  # Right STN iEEG during right hand
    data[subj]['meta'] = {'sfreq': 250.0}

consolidated_lcmv.npz:
    data[subj]['c']['L_STN_voxel']  # Left STN voxel method, eyes closed
    data[subj]['c']['R_STN_voxel']  # Right STN voxel method, eyes closed
    data[subj]['c']['STN_atlas']    # STN atlas method (bilateral), eyes closed
    data[subj]['l']['L_STN_voxel']  # Left STN voxel method, left hand
    data[subj]['l']['STN_atlas']    # STN atlas method, left hand
    data[subj]['meta'] = {'sfreq': 500.0}
"""

import numpy as np
from pathlib import Path
from tqdm import tqdm

# =============================================================================
# CONFIGURATION
# =============================================================================

IEEG_BASE = Path("/mnt/movement/users/jaizor/xtra/derivatives/integrated/ieeg")
LCMV_BASE = Path("/mnt/movement/users/jaizor/xtra/derivatives/integrated/lcmv")
OUTPUT_DIR = Path("/mnt/movement/users/jaizor/xtra/derivatives/integrated")

OUTPUT_IEEG = OUTPUT_DIR / "consolidated_ieeg.npz"
OUTPUT_LCMV = OUTPUT_DIR / "consolidated_lcmv.npz"

# Run codes - simple and consistent
RUN_CODES = ['c', 'o', 'l', 'r']  # eyes_closed, eyes_open, left_hand, right_hand
RUN_TO_COND = {'c': 'eyes_closed', 'o': 'eyes_open', 'l': 'left_hand', 'r': 'right_hand'}

# Valid subjects from your validation
VALID_SUBJECTS = ['sub-01', 'sub-05', 'sub-07', 'sub-10', 'sub-12', 'sub-14']

# =============================================================================
# IEEG CONSOLIDATION - INTUITIVE STRUCTURE
# =============================================================================

def consolidate_ieeg():
    """
    Creates consolidated_ieeg.npz with structure:
        data[subj][run][channel] = time_series
    
    Example:
        data = np.load('consolidated_ieeg.npz', allow_pickle=True)['data'].item()
        l_stn_ec = data['sub-01']['c']['L_STN']
        r_stn_lh = data['sub-05']['l']['R_STN']
    """
    print("\n🔍 Creating iEEG consolidation...")
    
    if not IEEG_BASE.exists():
        print(f"❌ iEEG folder not found: {IEEG_BASE}")
        return None

    # Create nested dictionary structure
    consolidated = {}
    
    # Process each valid subject
    for subj in tqdm(VALID_SUBJECTS, desc="Processing iEEG"):
        subj_path = IEEG_BASE / subj
        if not subj_path.exists():
            print(f"  ⚠️ {subj} folder not found, skipping")
            continue
        
        consolidated[subj] = {
            'meta': {
                'sfreq': 250.0,  # default
                'runs': []
            }
        }
        
        # Process each run
        for run in RUN_CODES:
            # Find iEEG file for this run
            files = list(subj_path.glob(f"*_{run}_ieeg.npz"))
            if not files:
                continue
                
            try:
                data = np.load(files[0], allow_pickle=True)
                meta = data['trial_metadata'].item()
                
                # Update sfreq from metadata
                consolidated[subj]['meta']['sfreq'] = meta.get('sfreq', 250.0)
                consolidated[subj]['meta']['runs'].append(run)
                
                # Initialize run dictionary
                consolidated[subj][run] = {}
                
                # Process each channel
                for key in data.files:
                    if key == 'trial_metadata':
                        continue
                    
                    # Map to standard channel names
                    channel = key
                    if 'STN' in channel:
                        if 'L' in channel or 'Left' in channel or 'left' in channel:
                            channel = 'L_STN'
                        elif 'R' in channel or 'Right' in channel or 'right' in channel:
                            channel = 'R_STN'
                    
                    consolidated[subj][run][channel] = data[key]
                    
            except Exception as e:
                print(f"  ⚠️ Error processing {subj} run {run}: {e}")
    
    # Save
    print(f"\n💾 Saving {OUTPUT_IEEG}...")
    np.savez_compressed(OUTPUT_IEEG, data=consolidated)
    print(f"✅ Saved iEEG data for {len(consolidated)} subjects")
    
    return consolidated

# =============================================================================
# LCMV CONSOLIDATION - INTUITIVE STRUCTURE
# =============================================================================

def consolidate_lcmv():
    """
    Creates consolidated_lcmv.npz with structure:
        data[subj][run]['L_STN_voxel'] = time_series
        data[subj][run]['R_STN_voxel'] = time_series
        data[subj][run]['STN_atlas'] = time_series
    
    Example:
        data = np.load('consolidated_lcmv.npz', allow_pickle=True)['data'].item()
        l_voxel_ec = data['sub-01']['c']['L_STN_voxel']
        atlas_lh = data['sub-05']['l']['STN_atlas']
    """
    print("\n🔍 Creating LCMV consolidation...")
    
    if not LCMV_BASE.exists():
        print(f"❌ LCMV folder not found: {LCMV_BASE}")
        return None

    # Find all ROI files
    all_files = list(LCMV_BASE.glob("*_rois.npz"))
    
    # Create nested dictionary structure
    consolidated = {}
    
    for file_path in tqdm(all_files, desc="Processing LCMV"):
        # Extract subject ID
        subj = file_path.stem.replace('_rois', '')
        
        # Only keep valid subjects
        if subj not in VALID_SUBJECTS:
            continue
        
        try:
            data = np.load(file_path, allow_pickle=True)
            
            consolidated[subj] = {
                'meta': {
                    'sfreq': 500.0,  # default
                    'runs': [],
                    'available': []
                }
            }
            
            # Get metadata
            if "metadata" in data.files:
                meta = data["metadata"].item()
                consolidated[subj]['meta']['sfreq'] = meta.get('sfreq', 500.0)
            
            # Initialize run dictionaries
            for run in RUN_CODES:
                consolidated[subj][run] = {}
            
            # Process each key
            for key in data.files:
                if key in ["metadata", "times"]:
                    continue
                
                # Parse the key: e.g., "L_STN_eyes_closed" or "STN_eyes_closed"
                parts = key.split('_')
                
                if len(parts) < 2:
                    continue
                
                # Determine the condition
                condition = '_'.join(parts[-2:]) if len(parts) > 2 else parts[-1]
                
                # Map condition to run code
                run = None
                for r, cond in RUN_TO_COND.items():
                    if cond == condition:
                        run = r
                        break
                
                if run is None:
                    continue
                
                # Determine ROI type and create clear name
                if parts[0] == 'L' and parts[1] == 'STN':
                    roi_name = 'L_STN_voxel'
                elif parts[0] == 'R' and parts[1] == 'STN':
                    roi_name = 'R_STN_voxel'
                elif parts[0] == 'STN':
                    roi_name = 'STN_atlas'
                else:
                    continue
                
                # Store with clear name
                consolidated[subj][run][roi_name] = data[key]
                
                # Track available ROIs
                if roi_name not in consolidated[subj]['meta']['available']:
                    consolidated[subj]['meta']['available'].append(roi_name)
                
                # Track runs
                if run not in consolidated[subj]['meta']['runs']:
                    consolidated[subj]['meta']['runs'].append(run)
            
        except Exception as e:
            print(f"  ⚠️ Error processing {subj}: {e}")
    
    # Save
    print(f"\n💾 Saving {OUTPUT_LCMV}...")
    np.savez_compressed(OUTPUT_LCMV, data=consolidated)
    print(f"✅ Saved LCMV data for {len(consolidated)} subjects")
    
    return consolidated

# =============================================================================
# VALIDATION FUNCTION
# =============================================================================

def verify_consolidation():
    """Verify that the consolidation worked as expected"""
    
    print("\n" + "="*70)
    print("🔍 VERIFYING CONSOLIDATION")
    print("="*70)
    
    # Load both files
    ieeg = np.load(OUTPUT_IEEG, allow_pickle=True)['data'].item()
    lcmv = np.load(OUTPUT_LCMV, allow_pickle=True)['data'].item()
    
    print(f"\n✅ Files loaded successfully")
    print(f"   iEEG subjects: {sorted(ieeg.keys())}")
    print(f"   LCMV subjects: {sorted(lcmv.keys())}")
    
    # Check sub-01 as example
    if 'sub-01' in ieeg and 'sub-01' in lcmv:
        print("\n📊 sub-01 data structure:")
        
        print("\n   🔵 iEEG:")
        print(f"      Metadata: sfreq={ieeg['sub-01']['meta']['sfreq']} Hz")
        print(f"      Runs available: {ieeg['sub-01']['meta']['runs']}")
        for run in ieeg['sub-01']['meta']['runs']:
            channels = list(ieeg['sub-01'][run].keys())
            print(f"      Run {run}: {channels}")
        
        print("\n   🔴 LCMV:")
        print(f"      Metadata: sfreq={lcmv['sub-01']['meta']['sfreq']} Hz")
        print(f"      Runs available: {lcmv['sub-01']['meta']['runs']}")
        print(f"      Available ROIs: {lcmv['sub-01']['meta']['available']}")
        for run in lcmv['sub-01']['meta']['runs']:
            rois = list(lcmv['sub-01'][run].keys())
            print(f"      Run {run}: {rois}")
    
    print("\n✅ Verification complete!")
    print("\n📘 ACCESS PATTERNS:")
    print('''
    # Load data
    ieeg = np.load('consolidated_ieeg.npz', allow_pickle=True)['data'].item()
    lcmv = np.load('consolidated_lcmv.npz', allow_pickle=True)['data'].item()
    
    # Get data for validation
    for subj in valid_subjects:
        # iEEG ground truth
        l_stn_true = ieeg[subj]['c']['L_STN']  # eyes closed
        l_stn_true_lh = ieeg[subj]['l']['L_STN']  # left hand
        
        # Voxel method
        l_stn_voxel = lcmv[subj]['c']['L_STN_voxel']
        l_stn_voxel_lh = lcmv[subj]['l']['L_STN_voxel']
        
        # Atlas method (bilateral)
        stn_atlas = lcmv[subj]['c']['STN_atlas']
        stn_atlas_lh = lcmv[subj]['l']['STN_atlas']
    ''')

# =============================================================================
# MAIN
# =============================================================================

if __name__ == "__main__":
    print("🚀 Starting Data Consolidation...")
    print(f"   iEEG source: {IEEG_BASE}")
    print(f"   LCMV source: {LCMV_BASE}")
    print(f"   Valid subjects: {VALID_SUBJECTS}")
    
    # Create consolidated files
    ieeg_data = consolidate_ieeg()
    lcmv_data = consolidate_lcmv()
    
    if ieeg_data and lcmv_data:
        verify_consolidation()
    
    print("\n" + "="*70)
    print("✅ CONSOLIDATION COMPLETE!")
    print("="*70)
    print(f"\n📁 Files created:")
    print(f"   - {OUTPUT_IEEG.name}")
    print(f"   - {OUTPUT_LCMV.name}")

🚀 Starting Data Consolidation...
   iEEG source: /mnt/movement/users/jaizor/xtra/derivatives/integrated/ieeg
   LCMV source: /mnt/movement/users/jaizor/xtra/derivatives/integrated/lcmv
   Valid subjects: ['sub-01', 'sub-05', 'sub-07', 'sub-10', 'sub-12', 'sub-14']

🔍 Creating iEEG consolidation...


Processing iEEG:   0%|          | 0/6 [00:00<?, ?it/s]

Processing iEEG: 100%|██████████| 6/6 [00:00<00:00, 135.06it/s]


💾 Saving /mnt/movement/users/jaizor/xtra/derivatives/integrated/consolidated_ieeg.npz...


✅ Saved iEEG data for 6 subjects

🔍 Creating LCMV consolidation...


Processing LCMV: 100%|██████████| 12/12 [00:00<00:00, 246.66it/s]


💾 Saving /mnt/movement/users/jaizor/xtra/derivatives/integrated/consolidated_lcmv.npz...


✅ Saved LCMV data for 6 subjects

🔍 VERIFYING CONSOLIDATION

✅ Files loaded successfully
   iEEG subjects: ['sub-01', 'sub-05', 'sub-07', 'sub-10', 'sub-12', 'sub-14']
   LCMV subjects: ['sub-01', 'sub-05', 'sub-07', 'sub-10', 'sub-12', 'sub-14']

📊 sub-01 data structure:

   🔵 iEEG:
      Metadata: sfreq=250.0 Hz
      Runs available: ['c', 'o', 'l', 'r']
      Run c: ['L_STN', 'ECOG-8-9-R', 'ECOG-10-11-R', 'ECOG-10-11-L', 'ECOG-8-9-L', 'R_STN']
      Run o: ['L_STN', 'ECOG-8-9-R', 'ECOG-10-11-R', 'ECOG-10-11-L', 'ECOG-8-9-L', 'R_STN']
      Run l: ['L_STN', 'ECOG-8-9-R', 'ECOG-10-11-R', 'ECOG-10-11-L', 'ECOG-8-9-L', 'R_STN']
      Run r: ['L_STN', 'ECOG-8-9-R', 'ECOG-10-11-R', 'ECOG-10-11-L', 'ECOG-8-9-L', 'R_STN']

   🔴 LCMV:
      Metadata: sfreq=500.0 Hz
      Runs available: ['c', 'l', 'o', 'r']
      Available ROIs: ['L_STN_voxel', 'R_STN_voxel', 'STN_atlas']
      Run c: ['L_STN_voxel', 'R_STN_voxel', 'STN_atlas']
      Run l: ['L_STN_voxel', 'R_STN_voxel', 'STN_atlas']
      R